# 14 - Kaggle runtime benchmark (KAGGLE numbers, NOT Jetson)
Spec notebook 14 / plan F6. Everything here is measured on the **Kaggle GPU / CPU this notebook runs on**; every table and plot carries that label. These are a reference for the report and a dry run of the Orin benchmark; **they are not Jetson metrics**, and no edge claim is made from them.

1. **Model latency**: YOLO11n / YOLO11s, PyTorch (ultralytics 8.3.40), on real test images already in memory (no file decode): GPU FP16 and FP32 at batch 1 (the edge case: one camera frame at a time), GPU FP16 at batch 16 (per image), CPU FP32 at batch 1. Warm-up, then p50 / p90 / p95 / p99 / mean, FPS, ultralytics per-stage times (pre-process, inference, post-process), peak GPU memory, and GPU power / temperature / clock from `nvidia-smi` while it runs.
2. **ONNX export check** (rehearsal of the Orin step G4): export both models, run them with ONNX Runtime on CPU, compare boxes and confidences with PyTorch CPU on the same images, and time them. TensorRT is **not** built here: engines are hardware-specific and are built on the Orin.
3. **End-to-end pipeline** (`src/runtime/pipeline.py`, the same per-frame code the Orin replay will use): decode, scene features, controller decision, detection, slot association, tracker + temporal filter, drift monitor, timed per stage on sampled test sequences for fixed YOLO11n, fixed YOLO11s and AdaptivePark under the deployed decision rule. A second AdaptivePark run with the **cached notebook 07 detections** checks that live detection reproduces the replay (a unit test proves the pipeline equals the replay given the same detections).
4. **Batch-1 cost ratio**: the YOLO11s / YOLO11n latency ratio at batch 1 vs the batched ratio 2.63 used as the compute unit in notebooks 10 and 13.

**Inputs (Add Input -> Your Work):** notebook 03 (images + splits), notebook 04 (YOLO11n weights), notebook 05 (YOLO11s weights), notebook 07, 09, 10, 11 outputs (full runs) and notebook 12 output (drift baselines; optional, the drift stage is skipped without it).
**Accelerator:** GPU T4 (x1 or x2; one GPU is used). Internet ON (repo, pinned ultralytics, ONNX packages).
**How to run:** `SMOKE = True` + Run All. Then `SMOKE = False` + **Save Version -> Save & Run All**. Run nothing else in the session while it measures.

In [ ]:
# --- Setup: get the project code onto Kaggle ---
# Preferred: attach the code as a Kaggle dataset (works for a private GitHub repo).
# Fallback: git clone (only works if the GitHub repo is public and Internet is ON).
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/Amith2277/Edge-Computing-EndSem.git"
REPO_DIR = "/kaggle/working/repo"
MARK = os.path.join("src", "common", "config.py")

def find_attached_code():
    for pat in ("/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"):
        for d in sorted(glob.glob(pat)):
            if os.path.isfile(os.path.join(d, MARK)):
                return d.rstrip("/")
    return None

if os.path.isdir(REPO_DIR):
    print("using existing", REPO_DIR)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):  # cloned earlier: fetch the latest code
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout.strip())
else:
    attached = find_attached_code()
    if attached:
        shutil.copytree(attached, REPO_DIR)
        print("copied attached code dataset:", attached)
    else:
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout)
        if r.returncode != 0:
            raise SystemExit("No attached code dataset found and git clone failed (private repo?). "
                             "Upload adaptivepark-code.zip as a Kaggle dataset and Add Input it, then re-run.")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("code at", REPO_DIR, "| files:", sorted(os.listdir(REPO_DIR))[:12])


In [ ]:
# Install the pinned ultralytics (same version as the Orin so the .pt loads there). No torch upgrade.
import importlib.metadata as md_
try:
    have = md_.version("ultralytics")
except md_.PackageNotFoundError:
    have = None
if have != "8.3.40":
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.3.40"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    print(r.stdout[-2000:])
# torch >= 2.6 defaults torch.load(weights_only=True), which ultralytics 8.3.40 checkpoints do not satisfy.
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")
# Training uses one GPU. Hiding the second one lets ultralytics use all 4 CPU cores for data loading
# (it divides the cores by the number of visible GPUs: 2 loader workers on "T4 x2"). Must be set before torch loads.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import torch, ultralytics
# Turn off experiment-tracker integrations (Kaggle's ray breaks the raytune callback; wandb can wait for a login).
# Runs before any trainer exists. If you saw a raytune error earlier, restart the kernel first.
from src.training import yolo_train as yt
print("tracker integrations:", yt.disable_integrations())
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
# Settings, inputs, hardware identity
SMOKE = True                      # True: few iterations / sequences. False: the full benchmark
N_IMAGES = 64                     # distinct test images cycled through the latency loops
WARMUP = 10 if SMOKE else 30
ITERS = {"gpu": 30 if SMOKE else 300, "cpu": 5 if SMOKE else 40}   # timed calls per configuration
BATCH_BIG = 16
PIPE_SEQUENCES = 2 if SMOKE else 6   # sampled test sequences for the end-to-end pipeline (stratified by site)
PIPE_MAX_FRAMES = 20 if SMOKE else 10 ** 9
ONNX_IMAGES = 4 if SMOKE else 20
LIGHT, HEAVY = "yolo11n", "yolo11s"
MAX_GAP_FACTOR = 3.0
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"

import os, json, time, csv, gzip, platform, threading, subprocess, shutil, gc
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import numpy as np
from PIL import Image
import torch
from src.common.config import load_yaml, config_hash
from src.common.status import new_run_id, write_status
from src.common.hashing import sha256_file
from src.training import yolo_train as yt
from src.evaluation import det_eval as de
from src.occupancy import cost_sensitive as cs
from src.temporal import filters as tf
from src.scene_analysis import features as sf
from src.slot_risk import risk as rk
from src.drift import monitor as dm
from src.runtime.pipeline import FramePipeline, STAGES
DCFG = load_yaml("configs/dataset.yaml"); FRAC = DCFG["dataset"]["subset_fraction"]
IMGSZ = {v: load_yaml("configs/train_{}.yaml".format(v))["imgsz"] for v in (LIGHT, HEAVY)}
OCFG, TCFG = load_yaml("configs/occupancy.yaml"), load_yaml("configs/temporal.yaml")

def find_one(name, folder, what, required=True):
    p = [x for x in yt.find_files(INPUT_ROOT, name) if x.parent.name == folder]
    assert p or not required, "{} not found: attach {}".format(name, what)
    return p[0] if p else None
man = find_one("split_manifest.csv", "splits", "notebook 03's output")
ROW_BY_ID = {int(r["image_id"]): r for r in de.read_manifest(man)}
H = json.loads((man.parent / "split_info.json").read_text())["manifest_hash"]
EXPORT = yt.find_export_dir(INPUT_ROOT, FRAC)
OCC = find_one("slot_scores_test.csv.gz", "occupancy", "notebook 07's output (full run)").parent
st07 = json.loads((OCC / "status.json").read_text()); assert st07["state"] == "COMPLETE"
R09 = json.loads(find_one("scene_risk_record.json", "scene_risk", "notebook 09's output").read_text())
R10 = json.loads(find_one("controller_record.json", "controller", "notebook 10's output").read_text())
R11 = json.loads(find_one("cost_sensitive_record.json", "cost_sensitive", "notebook 11's output").read_text())
for name, r in (("09", R09), ("10", R10), ("11", R11)):
    assert not r["smoke_test"] and r["dataset_split_hash"] == H and r["notebook07_run_id"] == st07["run_id"], "notebook {} output: smoke run or other split".format(name)
p12 = find_one("drift_record.json", "drift", "notebook 12's output", required=False)
R12 = json.loads(p12.read_text()) if p12 else None
if R12 is not None and (R12["smoke_test"] or R12["dataset_split_hash"] != H):
    print("notebook 12 output is a smoke run or another split: drift stage skipped"); R12 = None
WEIGHTS, TREC = {}, {}
ALL_TREC = yt.find_files(INPUT_ROOT, "training_record.json")
for v in (LIGHT, HEAVY):
    f = [p for p in ALL_TREC if p.parent.name == "{}_f{}".format(v, FRAC)]
    assert f, "{} weights not found: Add Input -> notebook {} output (folder {}_f{}); training records found: {}".format(
        v, "04" if v == LIGHT else "05", v, FRAC, [str(p.parent) for p in ALL_TREC] or "none")
    TREC[v] = json.loads(f[0].read_text()); WEIGHTS[v] = f[0].parent / "best.pt"
    assert sha256_file(WEIGHTS[v]) == TREC[v]["checkpoint_sha256"] and TREC[v]["dataset_split_hash"] == H, v

THR07, ASSOC = {}, {}
for v in (LIGHT, HEAVY):
    rec = json.loads((OCC / "occupancy_record_{}.json".format(v)).read_text())
    THR07[v] = float(rec["chosen_on_val"]["conf_thr"]); ASSOC[v] = rec["chosen_on_val"]
DEC = OCFG["decision"]; P_THR = cs.bayes_threshold(float(DEC["cost_false_vacant"]), float(DEC["cost_false_occupied"]))
METH = DEC["calibration"]["method"]
T_DEP = {v: cs.score_threshold_for(cs.from_params(R11["calibration"][v]["params"][METH]), P_THR) for v in (LIGHT, HEAVY)}
FILT = dict(TCFG["deployed"][LIGHT])
SCALES = R09["difficulty"]["scales"]
CTX = {"thr": T_DEP, "filter": FILT, "wD": np.array([R09["difficulty"]["weights"][f] for f in sf.FEATURES]),
       "wR": np.array([R09["slot_risk"]["weights"][c] for c in rk.COMPONENTS]), "agg": R09["slot_risk"]["aggregation"],
       "age_ref_s": float(R09["slot_risk"]["max_age_s_for_A"]), "inst_frames": int(R09["instability_frames"]),
       "occ_scale": float(SCALES["occlusion"]), "light": LIGHT, "heavy": HEAVY}
ACC_CFG = R10["chosen"]["adaptive"]["accuracy_point"]
SLOTS = json.loads((OCC / "slots.json").read_text())
CAM_SLOTS = defaultdict(list)
for s in SLOTS:
    CAM_SLOTS[s["camera"]].append(s)

def sh(cmd):
    try:
        return subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True, timeout=20).stdout.strip()
    except Exception as e:
        return "unavailable ({})".format(type(e).__name__)
GPU = torch.cuda.is_available()
cpu_model = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), platform.processor()) if os.path.exists("/proc/cpuinfo") else platform.processor()
HW = {"platform": "Kaggle", "gpu": torch.cuda.get_device_name(0) if GPU else None, "gpu_count": torch.cuda.device_count(),
      "driver": sh(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"]).splitlines()[0] if GPU else None,
      "cuda": torch.version.cuda, "torch": torch.__version__, "cpu": cpu_model, "cpu_cores": os.cpu_count(), "python": platform.python_version(),
      "ultralytics": __import__("ultralytics").__version__}
LABEL = "Kaggle {} / {} ({} cores) - NOT Jetson".format(HW["gpu"] or "no GPU", HW["cpu"], HW["cpu_cores"])
print(json.dumps(HW, indent=1)); print("label:", LABEL)
print("deployed decision: score >=", {k: round(x, 4) for k, x in T_DEP.items()}, "+", tf.label(FILT), "| drift stage:", "on" if R12 else "off")
OUT = Path(WORK) / "artifacts" / "benchmark_kaggle"
OUT.mkdir(parents=True, exist_ok=True)
SUF = "_smoke" if SMOKE else ""
CFG_HASH = config_hash({"smoke": SMOKE, "n_images": N_IMAGES, "warmup": WARMUP, "iters": ITERS, "batch": BATCH_BIG, "pipe": PIPE_SEQUENCES,
                        "split": H, "weights": {v: TREC[v]["checkpoint_sha256"] for v in TREC}, "nb10": R10["run_id"], "nb11": R11["run_id"]})
RUN_ID = new_run_id("bmk")
STATUS = OUT / "status.json"
write_status(STATUS, "14_edge_benchmark_kaggle", RUN_ID, "RUNNING", CFG_HASH, "inputs loaded")

In [ ]:
# 1. Model latency (PyTorch): images in memory, warm-up, timed calls; nvidia-smi sampled in the background
from ultralytics import YOLO
MODELS = {v: YOLO(str(WEIGHTS[v])) for v in (LIGHT, HEAVY)}
names_test = sorted(os.listdir(str(EXPORT / "images" / "test")))
pick = [names_test[i] for i in np.linspace(0, len(names_test) - 1, min(N_IMAGES, len(names_test))).astype(int)]
IMGS = []
for n_ in pick:
    with Image.open(str(EXPORT / "images" / "test" / n_)) as im:
        IMGS.append(np.asarray(im.convert("RGB"))[:, :, ::-1].copy())        # BGR, as ultralytics expects for arrays
print("{} test images in memory, size {}".format(len(IMGS), IMGS[0].shape))

class GpuSampler(object):
    """nvidia-smi every 200 ms while a configuration runs (board power, temperature, SM clock, utilisation, memory)."""
    Q = "power.draw,temperature.gpu,clocks.sm,utilization.gpu,memory.used"
    def __init__(self):
        self.rows, self.p = [], None
    def __enter__(self):
        if GPU and shutil.which("nvidia-smi"):
            self.p = subprocess.Popen(["nvidia-smi", "-i", "0", "--query-gpu=" + self.Q, "--format=csv,noheader,nounits", "-lms", "200"],
                                      stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, universal_newlines=True)
            self.th = threading.Thread(target=self._read, daemon=True); self.th.start()
        return self
    def _read(self):
        for line in self.p.stdout:
            try:
                self.rows.append([float(x) for x in line.strip().split(",")])
            except ValueError:
                pass
    def __exit__(self, *a):
        if self.p:
            self.p.terminate(); self.p.wait(timeout=5)
    def summary(self):
        if not self.rows:
            return None
        a = np.array(self.rows)
        return {"samples": len(a), "power_w_mean": round(float(a[:, 0].mean()), 1), "power_w_max": round(float(a[:, 0].max()), 1),
                "temp_c_max": float(a[:, 1].max()), "sm_clock_mhz_median": float(np.median(a[:, 2])), "util_pct_mean": round(float(a[:, 3].mean()), 1),
                "mem_used_mib_max": float(a[:, 4].max())}

def pct(x):
    x = np.asarray(x, float)
    return {"p50": round(float(np.percentile(x, 50)), 3), "p90": round(float(np.percentile(x, 90)), 3), "p95": round(float(np.percentile(x, 95)), 3),
            "p99": round(float(np.percentile(x, 99)), 3), "mean": round(float(x.mean()), 3), "n": int(len(x))}

def check_backend(m, device, half):
    # fail loudly if the model is not on the device / precision the label claims
    be = getattr(getattr(m, "predictor", None), "model", None)
    if be is None or not hasattr(be, "fp16"):
        return                                              # stand-in detector in dry runs
    on_cpu = getattr(be, "device", None) is not None and be.device.type == "cpu"
    assert bool(be.fp16) == bool(half) and on_cpu == (device == "cpu"), "measured {} fp16={} but the label says {} half={}".format(
        be.device, be.fp16, device, half)

def bench(v, device, half, batch, iters):
    # a FRESH model per configuration: ultralytics sets device and FP16 once, at the first predict call, and ignores
    # later device / half arguments (the first full run measured GPU FP16 under the FP32 and CPU labels)
    m = YOLO(str(WEIGHTS[v]))
    kw = dict(conf=0.05, iou=0.7, imgsz=IMGSZ[v], device=device, half=half, max_det=300, verbose=False)
    batches = [[IMGS[(i * batch + j) % len(IMGS)] for j in range(batch)] for i in range(WARMUP + iters)]
    for b in batches[:WARMUP]:
        m.predict(b, **kw)
    check_backend(m, device, half)
    if GPU:
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    wall, stage = [], defaultdict(list)
    with GpuSampler() as gs:
        for b in batches[WARMUP:]:
            t0 = time.perf_counter()
            res = m.predict(b, **kw)
            out = [r.boxes.data.cpu().numpy() for r in res]          # results on the host, as the pipeline uses them
            if GPU:
                torch.cuda.synchronize()
            wall.append(1000.0 * (time.perf_counter() - t0))
            for k_, x in res[0].speed.items():
                stage[k_].append(x)
    w = pct(wall); per_img = pct(np.array(wall) / batch)
    return {"model": v, "device": "cuda:0" if device != "cpu" else "cpu", "precision": "fp16" if half else "fp32", "batch": batch,
            "imgsz": IMGSZ[v], "call_ms": w, "per_image_ms": per_img, "fps_images": round(1000.0 / per_img["mean"], 1),
            "ultralytics_stage_ms_per_image": {k_: round(float(np.median(x)), 3) for k_, x in stage.items()},
            "gpu_peak_mem_mib": round(torch.cuda.max_memory_allocated() / 2 ** 20, 1) if GPU and device != "cpu" else None,
            "nvidia_smi": gs.summary() if device != "cpu" else None, "warmup": WARMUP, "label": LABEL}

CONFIGS = []
if GPU:
    CONFIGS += [(v, 0, True, 1, ITERS["gpu"]) for v in (LIGHT, HEAVY)] + [(v, 0, False, 1, ITERS["gpu"]) for v in (LIGHT, HEAVY)]
    CONFIGS += [(v, 0, True, BATCH_BIG, max(10, ITERS["gpu"] // 10)) for v in (LIGHT, HEAVY)]
CONFIGS += [(v, "cpu", False, 1, ITERS["cpu"]) for v in (LIGHT, HEAVY)]
# Warm both models on the GPU before any timing: in the smoke run the first configuration ran at a low GPU clock
# (585 MHz) and its mean was inflated by start-up outliers, which made YOLO11s look faster than YOLO11n.
if GPU:
    for v in (LIGHT, HEAVY):
        for i in range(3 * WARMUP):
            MODELS[v].predict(IMGS[i % len(IMGS)], conf=0.05, iou=0.7, imgsz=IMGSZ[v], device=0, half=True, max_det=300, verbose=False)
    torch.cuda.synchronize()
LAT = []
try:
    for cfg in CONFIGS:
        r = bench(*cfg); LAT.append(r)
        if r["nvidia_smi"] and r["nvidia_smi"]["power_w_mean"]:
            r["energy_j_per_image_board"] = round(r["nvidia_smi"]["power_w_mean"] * r["per_image_ms"]["mean"] / 1000.0, 4)
        print("{:<8} {:<6} {:<5} b{:<3} per image p50 {:>8.3f} p95 {:>8.3f} p99 {:>8.3f} mean {:>8.3f} ms | {:>7.1f} img/s | stages {} | GPU mem {} MiB | smi {}".format(
            r["model"], r["device"], r["precision"], r["batch"], r["per_image_ms"]["p50"], r["per_image_ms"]["p95"], r["per_image_ms"]["p99"],
            r["per_image_ms"]["mean"], r["fps_images"], r["ultralytics_stage_ms_per_image"], r["gpu_peak_mem_mib"], r["nvidia_smi"]))
except Exception as e:
    write_status(STATUS, "14_edge_benchmark_kaggle", RUN_ID, "FAILED", CFG_HASH, repr(e)[:500]); raise
def lat(v, dev, prec, b):
    r = [x for x in LAT if x["model"] == v and x["device"].startswith(dev) and x["precision"] == prec and x["batch"] == b]
    return r[0] if r else None
RATIO = {}
for key, dev, prec, b in (("gpu_fp16_b1", "cuda", "fp16", 1), ("gpu_fp32_b1", "cuda", "fp32", 1), ("gpu_fp16_b{}".format(BATCH_BIG), "cuda", "fp16", BATCH_BIG), ("cpu_fp32_b1", "cpu", "fp32", 1)):
    a_, b_ = lat(LIGHT, dev, prec, b), lat(HEAVY, dev, prec, b)
    if a_ and b_:
        RATIO[key] = round(b_["per_image_ms"]["p50"] / a_["per_image_ms"]["p50"], 3)         # medians: robust to outliers
print("\nYOLO11s / YOLO11n median latency ratio ({}):".format(LABEL), RATIO, "| notebooks 10 / 13 used 2.63 (batched, notebook 06)")

In [ ]:
# 2. ONNX export + ONNX Runtime (CPU) parity and latency vs PyTorch CPU. TensorRT is built on the Orin, not here.
def match(a, b, thr=0.5):
    """Greedy IoU matching of two (N, 6) box arrays [x0, y0, x1, y1, conf, cls]: matched share, conf and coordinate differences."""
    if not len(a) and not len(b):
        return 1.0, [], []
    if not len(a) or not len(b):
        return 0.0, [], []
    if True:
        x0 = np.maximum(a[:, None, 0], b[None, :, 0]); y0 = np.maximum(a[:, None, 1], b[None, :, 1])
        x1 = np.minimum(a[:, None, 2], b[None, :, 2]); y1 = np.minimum(a[:, None, 3], b[None, :, 3])
        inter = np.clip(x1 - x0, 0, None) * np.clip(y1 - y0, 0, None)
        ar = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
        iou = inter / (ar(a)[:, None] + ar(b)[None, :] - inter + 1e-9)
    used, dc, dx = set(), [], []
    for i in np.argsort(-a[:, 4]):
        j = int(np.argmax(np.where([k in used for k in range(len(b))], -1, iou[i])))
        if iou[i, j] >= thr and j not in used:
            used.add(j); dc.append(abs(float(a[i, 4] - b[j, 4]))); dx.append(float(np.abs(a[i, :4] - b[j, :4]).max()))
    return len(used) / float(max(len(a), len(b))), dc, dx

ONNX = {}
for v in (LIGHT, HEAVY):
    try:
        wdir = OUT / "onnx" / v; wdir.mkdir(parents=True, exist_ok=True)
        wcopy = wdir / "best.pt"; shutil.copy(str(WEIGHTS[v]), str(wcopy))          # inputs are read-only; export next to a copy
        t0 = time.time()
        # torch >= 2.9 exports with the dynamo exporter by default (needs onnxscript). The Orin's torch 2.1 uses the
        # TorchScript exporter, so the rehearsal uses that one too.
        import inspect, sys
        if "dynamo" in inspect.signature(torch.onnx.export).parameters and not getattr(torch.onnx.export, "_legacy", False):
            _orig_export = torch.onnx.export
            def _export_legacy(*a, **k):
                k.setdefault("dynamo", False)
                return _orig_export(*a, **k)
            _export_legacy._legacy = True
            torch.onnx.export = _export_legacy
        try:
            import onnxruntime  # noqa: F401
        except ImportError:
            print(subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime"], stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, universal_newlines=True).stdout[-500:])
        path = YOLO(str(wcopy)).export(format="onnx", imgsz=IMGSZ[v], opset=12, dynamic=False, simplify=False, half=False, device="cpu")
        exp_s = time.time() - t0
        om = YOLO(str(path), task="detect")
        kw = dict(conf=0.05, iou=0.7, imgsz=IMGSZ[v], device="cpu", max_det=300, verbose=False)
        pt_cpu = YOLO(str(WEIGHTS[v]))          # fresh model: ultralytics fixes device and precision at its first predict call
        share, dcs, dxs, t_onnx, t_pt, share_op = [], [], [], [], [], []
        for im in IMGS[:ONNX_IMAGES]:
            t1 = time.perf_counter(); ro = om.predict(im, **kw)[0].boxes.data.cpu().numpy(); t_onnx.append(1000 * (time.perf_counter() - t1))
            t1 = time.perf_counter(); rp_ = pt_cpu.predict(im, half=False, **kw)[0].boxes.data.cpu().numpy(); t_pt.append(1000 * (time.perf_counter() - t1))
            s_, c_, x_ = match(rp_, ro); share.append(s_); dcs += c_; dxs += x_
            share_op.append(match(rp_[rp_[:, 4] >= THR07[v]], ro[ro[:, 4] >= THR07[v]])[0])   # boxes at the operating score threshold
        check_backend(pt_cpu, "cpu", False)
        ONNX[v] = {"status": "OK", "file": str(path), "sha256": sha256_file(path), "size_mb": round(os.path.getsize(path) / 2 ** 20, 2),
                   "export_s": round(exp_s, 1), "opset": 12, "images": ONNX_IMAGES, "box_match_share_mean": round(float(np.mean(share)), 4),
                   "box_match_share_min": round(float(np.min(share)), 4), "box_match_share_at_operating_threshold": round(float(np.mean(share_op)), 4),
                   "conf_abs_diff_max": round(float(np.max(dcs)), 5) if dcs else None,
                   "coord_abs_diff_max_px": round(float(np.max(dxs)), 3) if dxs else None,
                   "onnxruntime_cpu_ms": pct(t_onnx[1:] or t_onnx), "pytorch_cpu_ms_same_images": pct(t_pt[1:] or t_pt), "label": LABEL}
        try:
            import onnxruntime as ort; ONNX[v]["onnxruntime"] = ort.__version__
        except Exception:
            pass
    except Exception as e:
        ONNX[v] = {"status": "NOT RUN", "reason": repr(e)[:300]}
    print(v, json.dumps(ONNX[v], indent=1))

In [ ]:
# 3. End-to-end pipeline on sampled test sequences: live detection, per-stage wall time, accuracy, parity with cached detections
def parse_ts(s):
    return datetime.fromisoformat(s.replace("Z", "")).timestamp()
frames = defaultdict(lambda: {"sids": [], "valid": [], "gt": []})
with gzip.open(str(OCC / "slot_scores_test.csv.gz"), "rt") as f:
    for r in csv.DictReader(f):
        fr = frames[r["image"]]; fr.update(camera=r["camera"], ts=r["timestamp"], image_id=int(r["image_id"]))
        fr["sids"].append(r["slot_id"]); fr["valid"].append(int(r["valid"])); fr["gt"].append(int(r["gt_occupied"]))
names = list(frames)
cam = np.array([frames[n_]["camera"] for n_ in names]); sec = np.array([parse_ts(frames[n_]["ts"]) for n_ in names])
day = np.array([ROW_BY_ID[frames[n_]["image_id"]]["day"] for n_ in names])
sp = tf.split_sequences(cam, sec, day, MAX_GAP_FACTOR)
SEQS = []
for q in sorted(set(sp["seq"].tolist())):
    idx = np.nonzero(sp["seq"] == q)[0]; idx = idx[np.argsort(sec[idx], kind="stable")]
    SEQS.append({"camera": cam[idx[0]], "site": ROW_BY_ID[frames[names[idx[0]]]["image_id"]]["site"], "names": [names[i] for i in idx], "sec": sec[idx]})
cand = sorted(range(len(SEQS)), key=lambda i: (SEQS[i]["site"], i))
by_site = defaultdict(list)
for i in cand:
    by_site[SEQS[i]["site"]].append(i)
CHOSEN_SEQ = []
while len(CHOSEN_SEQ) < min(PIPE_SEQUENCES, len(SEQS)):
    for s_ in sorted(by_site):
        if by_site[s_] and len(CHOSEN_SEQ) < PIPE_SEQUENCES:
            lst = by_site[s_]; CHOSEN_SEQ.append(lst.pop(len(lst) // 2))
print("pipeline sequences:", [(SEQS[i]["camera"], len(SEQS[i]["names"])) for i in CHOSEN_SEQ])

CACHED = {}
for v in (LIGHT, HEAVY):
    z = np.load(str(OCC / "detections" / "{}_test.npz".format(v)))
    CACHED[v] = {"dets": z["dets"], "starts": z["starts"], "row": {str(n_): i for i, n_ in enumerate(z["names"])}}
CUR = {"name": None}
def live_detector(v, image):
    r = MODELS[v].predict(np.asarray(image)[:, :, ::-1], conf=0.05, iou=0.7, imgsz=IMGSZ[v], device=0 if GPU else "cpu", half=GPU, max_det=300, verbose=False)[0]
    return r.boxes.data.cpu().numpy()[:, :5]
def cached_detector(v, image):
    D = CACHED[v]; k = D["row"][CUR["name"]]
    return D["dets"][D["starts"][k]:D["starts"][k + 1]][:, :5]

DRIFT = None
if R12:
    BASEL = json.loads((p12.parent / "baselines.json").read_text())
    DRIFT = {"test": R12["test_chosen_on_val"], "thr": R12["thresholds"][R12["test_chosen_on_val"]], "window": int(R12["window_frames"]),
             "base": {c: {s: (np.array(b[s], float) if b.get(s) is not None else None) for s in dm.SIGNALS} for c, b in BASEL.items()}}
def make_drift(camera):
    if not DRIFT or camera not in DRIFT["base"]:
        return None
    return dm.DriftMonitor(camera, DRIFT["base"][camera], DRIFT["thr"], DRIFT["test"], DRIFT["window"])

def run_pipe(i, strategy, detector):
    q = SEQS[i]
    pipe = FramePipeline(CAM_SLOTS[q["camera"]], CTX, strategy, ACC_CFG if strategy == "adaptive" else {}, ASSOC, THR07, detector,
                         motion_scale=float(SCALES["motion"]), drift=make_drift(q["camera"]))
    rows = []
    for t, n_ in enumerate(q["names"][:PIPE_MAX_FRAMES]):
        fr = frames[n_]; CUR["name"] = n_
        t0 = time.perf_counter()
        with Image.open(str(EXPORT / "images" / "test" / n_)) as im:
            img = im.convert("RGB"); img.load()
        dec_ms = 1000 * (time.perf_counter() - t0)
        o = pipe.step(img, float(q["sec"][t]), sids=fr["sids"], valid=np.array(fr["valid"], bool), frame=n_)
        g = np.array(fr["gt"])[np.array(fr["valid"], bool)]
        k = o["state"] >= 0
        rows.append({"decode": dec_ms, **o["ms"], "total": dec_ms + sum(o["ms"].values()), "model": o["model"], "mode": o["mode"],
                     "state": o["state"].copy(), "right": int((o["state"][k] == g[k]).sum()), "n": int(k.sum()),
                     "alert": None if not o["drift"] else o["drift"]["severity"]})
    return rows

# warm the OS file cache for every frame first: in the smoke run the first strategy paid a cold read from /kaggle/input
# (decode 75 ms vs 7 ms afterwards). Decode is reported with a warm file cache.
for i in CHOSEN_SEQ:
    for n_ in SEQS[i]["names"][:PIPE_MAX_FRAMES]:
        with open(str(EXPORT / "images" / "test" / n_), "rb") as fh:
            fh.read()
PIPE = {}
try:
    for strategy, det_name in (("fixed_n", "live"), ("fixed_s", "live"), ("adaptive", "live"), ("adaptive", "cached nb07 detections")):
        rows = []
        for i in CHOSEN_SEQ:
            rows += [dict(r, seq=i) for r in run_pipe(i, strategy, live_detector if det_name == "live" else cached_detector)]
        PIPE[(strategy, det_name)] = rows
except Exception as e:
    write_status(STATUS, "14_edge_benchmark_kaggle", RUN_ID, "FAILED", CFG_HASH, repr(e)[:500]); raise

PIPE_SUM = {}
print("\nEnd-to-end per frame, ms ({}; batch 1, {} detection, deployed decision rule)".format(LABEL, "GPU FP16" if GPU else "CPU FP32"))
print("  {:<36} {:>6} {:>8} {:>8} {:>8} {:>8} | {}".format("strategy | detector", "frames", "p50", "p95", "mean", "FPS", "mean per stage"))
for key, rows in PIPE.items():
    tot = [r["total"] for r in rows]
    s_ = {"frames": len(rows), "total_ms": pct(tot), "fps": round(1000.0 / np.mean(tot), 1),
          "stage_mean_ms": {k_: round(float(np.mean([r[k_] for r in rows])), 3) for k_ in ("decode",) + STAGES},
          "detect_ms_executed_frames": pct([r["detect"] for r in rows if r["model"]]) if any(r["model"] for r in rows) else None,
          "calls": {m_: sum(r["model"] == m_ for r in rows) for m_ in (LIGHT, HEAVY)}, "skipped": sum(r["model"] is None for r in rows),
          "accuracy": round(sum(r["right"] for r in rows) / max(1, sum(r["n"] for r in rows)), 4),
          "drift_alert_frames": sum(r["alert"] not in (None, "ok") for r in rows), "label": LABEL}
    PIPE_SUM["{} | {}".format(*key)] = s_
    print("  {:<36} {:>6} {:>8} {:>8} {:>8} {:>8} | {}".format("{} | {}".format(*key), s_["frames"], s_["total_ms"]["p50"], s_["total_ms"]["p95"],
          s_["total_ms"]["mean"], s_["fps"], s_["stage_mean_ms"]))
L, C_ = PIPE[("adaptive", "live")], PIPE[("adaptive", "cached nb07 detections")]
PARITY = {"frames": len(L), "same_model_choice": round(float(np.mean([a["model"] == b["model"] for a, b in zip(L, C_)])), 4),
          "same_slot_state": round(float(np.mean(np.concatenate([a["state"] == b["state"] for a, b in zip(L, C_)]))), 5)}
print("\nAdaptivePark live vs cached detections (same pipeline): model choice equal on {:.2%} of frames, slot state equal on {:.3%} of slot-frames".format(
      PARITY["same_model_choice"], PARITY["same_slot_state"]))
for k_, s_ in PIPE_SUM.items():
    print("  {:<36} calls {} skipped {} | accuracy {} | drift alert frames {}".format(k_, s_["calls"], s_["skipped"], s_["accuracy"], s_["drift_alert_frames"]))

In [ ]:
# 4. What the batch-1 latencies mean for the compute comparison (derived, Kaggle only)
DERIVED = None
b1 = (lat(LIGHT, "cuda", "fp16", 1), lat(HEAVY, "cuda", "fp16", 1)) if GPU else (lat(LIGHT, "cpu", "fp32", 1), lat(HEAVY, "cpu", "fp32", 1))
if all(b1):
    tn, ts = b1[0]["per_image_ms"]["p50"], b1[1]["per_image_ms"]["p50"]
    DERIVED = {"basis": "{} batch 1 median per-image latency".format("GPU FP16" if GPU else "CPU FP32"), "yolo11n_ms": tn, "yolo11s_ms": ts,
               "ratio_batch1": round(ts / tn, 3), "ratio_used_in_nb10_nb13": R10["cost_units"].get(HEAVY), "strategies": {}}
    for name in ("fixed YOLO11n", "fixed YOLO11s", "AdaptivePark (accuracy point)", "AdaptivePark (efficiency point)"):
        m_ = R10["test"].get(name)
        if m_:
            calls = m_["calls"]
            det = (calls.get(LIGHT, 0) * tn + calls.get(HEAVY, 0) * ts) / m_["frames"]
            DERIVED["strategies"][name] = {"detector_ms_per_frame": round(det, 3), "relative_to_fixed_n": round(det / tn, 3),
                                           "compute_units_nb10": m_["cost_per_frame"]}
    print("Derived from notebook 10 test call counts x {} ({}):".format(DERIVED["basis"], LABEL))
    print("  YOLO11s / YOLO11n at batch 1 = {} (notebooks 10 / 13 used {} from batched timing)".format(DERIVED["ratio_batch1"], DERIVED["ratio_used_in_nb10_nb13"]))
    for k_, x in DERIVED["strategies"].items():
        print("  {:<34} detector {:>7.3f} ms/frame = {:.3f} x fixed n   (nb10 compute units {})".format(k_, x["detector_ms_per_frame"], x["relative_to_fixed_n"], x["compute_units_nb10"]))

In [ ]:
# Plots (every title carries the hardware label)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))
lab = ["{} {} {} b{}".format(r["model"].replace("yolo11", "11"), r["device"].split(":")[0], r["precision"], r["batch"]) for r in LAT]
ax[0].barh(range(len(LAT)), [r["per_image_ms"]["mean"] for r in LAT], xerr=[[0] * len(LAT), [r["per_image_ms"]["p95"] - r["per_image_ms"]["mean"] for r in LAT]])
ax[0].set_yticks(range(len(LAT))); ax[0].set_yticklabels(lab, fontsize=8); ax[0].set_xscale("log")
ax[0].set(xlabel="ms per image (mean; whisker to p95), log scale", title="Model latency, PyTorch\n" + LABEL); ax[0].grid(alpha=0.3, axis="x")
keys = list(PIPE_SUM); bottom = np.zeros(len(keys))
for st in ("decode",) + STAGES:
    vals = np.array([PIPE_SUM[k_]["stage_mean_ms"][st] for k_ in keys]); ax[1].barh(range(len(keys)), vals, left=bottom, label=st); bottom += vals
ax[1].set_yticks(range(len(keys))); ax[1].set_yticklabels(keys, fontsize=8)
ax[1].set(xlabel="mean ms per frame", title="End-to-end pipeline, per stage\n" + LABEL); ax[1].legend(fontsize=7); ax[1].grid(alpha=0.3, axis="x")
plt.tight_layout(); plt.savefig(str(OUT / "kaggle_benchmark{}.png".format(SUF)), dpi=110); plt.show()

In [ ]:
# Record, environment, status
from src.common.env_record import save_environment
def jclean(x):
    if isinstance(x, dict):
        return {str(k): jclean(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [jclean(v) for v in x]
    if isinstance(x, (float, np.floating)):
        return float(x) if np.isfinite(x) else None
    if isinstance(x, np.ndarray):
        return jclean(x.tolist())
    if isinstance(x, (np.integer, np.bool_)):
        return x.item()
    return x
RECORD = {"run_id": RUN_ID, "config_hash": CFG_HASH, "smoke_test": SMOKE, "dataset_split_hash": H, "hardware": HW, "label": LABEL,
          "NOT_JETSON": "Kaggle measurements; the Orin benchmark (plan G7) replaces them for every edge claim",
          "weights_sha256": {v: TREC[v]["checkpoint_sha256"] for v in TREC}, "settings": {"warmup": WARMUP, "iters": ITERS, "images": len(IMGS),
          "batch_big": BATCH_BIG, "conf": 0.05, "nms_iou": 0.7, "imgsz": IMGSZ}, "model_latency": LAT, "latency_ratio_s_over_n": RATIO,
          "onnx": ONNX, "pipeline": PIPE_SUM, "pipeline_sequences": [{"camera": SEQS[i]["camera"], "frames": min(len(SEQS[i]["names"]), PIPE_MAX_FRAMES)} for i in CHOSEN_SEQ],
          "pipeline_parity_live_vs_cached": PARITY, "derived_compute": DERIVED,
          "energy_note": "energy_j_per_image_board = nvidia-smi board power x latency on a shared Kaggle GPU; includes idle power; not a Jetson energy figure",
          "limitation": "shared cloud machine; timings vary between sessions; TensorRT not built (Orin only)"}
(OUT / "benchmark_record{}.json".format(SUF)).write_text(json.dumps(jclean(RECORD), indent=2))
save_environment(str(OUT / "environment.json"))
write_status(STATUS, "14_edge_benchmark_kaggle", RUN_ID, "COMPLETE", CFG_HASH, "smoke={}".format(SMOKE),
             extra=jclean({"label": LABEL, "ratio": RATIO, "pipeline_fps": {k_: s_["fps"] for k_, s_ in PIPE_SUM.items()}}))
print("COMPLETE | saved to", OUT, sorted(p.name for p in OUT.iterdir()))

## What to send back
Download the notebook after **Save & Run All** (SMOKE = False) and send it.
`artifacts/benchmark_kaggle/` (latency record, ONNX files for comparison with the Orin export) feeds notebook 15, so **Save Version** with outputs.